# Create institution lineage works counts

Counts each institution's works three ways, so institution pages can show all three (oxjob #1823; INRAE's
curators were hand-linking strings that already counted for INRAE through its units and INRA, because only the
direct count was shown):

- `works_count` (built in CreateInstitutionsApi): works linked to the institution itself
- `works_count_with_children`: plus works linked to its children, all levels (ROR `parent` edges)
- `works_count_with_children_and_predecessors`: plus works linked to institutions it replaced and their children,
  i.e. every institution whose lineage holds it. Equals the `authorships.institutions.lineage` filter's count with
  `include_xpac=true`.

Super-system institutions are dropped as ancestors, as in the API's `lineage`.

Uses:
- `openalex.institutions.institutions`, `openalex.institutions.ror_relationships`, `openalex.institutions.institution_ancestors`
- `openalex.works.openalex_works`

Creates:
- `openalex.institutions.institution_lineage_works_counts` (institution_id, works_count_with_children,
  works_count_with_children_and_predecessors), joined into `institutions_api` by CreateInstitutionsApi.


In [ ]:
%run ../utils/variables

In [ ]:
%sql
-- Works counts by family for every institution (oxjob #1823). Three counts:
--   works_count                                 (CreateInstitutionsApi): works linked to the institution itself
--   works_count_with_children                   (+ works linked to its children, all levels, through ROR parent edges)
--   works_count_with_children_and_predecessors  (+ works linked to institutions it replaced, and their children:
--                                                every institution whose lineage holds it; equals the
--                                                authorships.institutions.lineage filter with include_xpac=true)
-- Super-system institutions are dropped as ancestors, as in the API's lineage.
CREATE OR REPLACE TABLE openalex.institutions.institution_lineage_works_counts AS
WITH RECURSIVE inst AS (
  SELECT id, REPLACE(ror_id, 'https://ror.org/', '') AS ror_id
  FROM openalex.institutions.institutions
  WHERE merge_into_id IS NULL
),
parent_edges AS (
  SELECT ror_id, related_ror_id
  FROM openalex.institutions.ror_relationships
  WHERE relationship_type = 'parent'
),
parent_path AS (
  SELECT i.id AS institution_id, p.id AS ancestor_id, p.ror_id AS ancestor_ror_id, 1 AS generation, ARRAY(i.id) AS path
  FROM inst i
  JOIN parent_edges e ON i.ror_id = e.ror_id
  JOIN inst p ON e.related_ror_id = p.ror_id
  UNION ALL
  SELECT pp.institution_id, g.id, g.ror_id, pp.generation + 1, ARRAY_UNION(pp.path, ARRAY(g.id))
  FROM parent_path pp
  JOIN parent_edges e ON pp.ancestor_ror_id = e.ror_id
  JOIN inst g ON e.related_ror_id = g.ror_id
  WHERE pp.generation < 10
    AND NOT ARRAY_CONTAINS(pp.path, g.id)
),
child_of AS (
  SELECT DISTINCT institution_id, ancestor_id
  FROM parent_path
  WHERE NOT ARRAY_CONTAINS(SUPER_SYSTEM_INSTITUTIONS, ancestor_id)
),
lineage_of AS (
  SELECT DISTINCT institution_id, ancestor_id
  FROM openalex.institutions.institution_ancestors
  LATERAL VIEW EXPLODE(FILTER(lineage_ids, x -> NOT ARRAY_CONTAINS(SUPER_SYSTEM_INSTITUTIONS, x))) t AS ancestor_id
),
au AS (
  SELECT w.id AS work_id, EXPLODE(w.authorships) AS a
  FROM openalex.works.openalex_works w
),
pairs AS (
  SELECT DISTINCT au.work_id, CAST(REPLACE(inst.id, 'https://openalex.org/I', '') AS BIGINT) AS institution_id
  FROM au LATERAL VIEW EXPLODE(au.a.institutions) t AS inst
  WHERE inst.id IS NOT NULL
),
expanded AS (
  SELECT work_id, institution_id AS counted_id, TRUE AS via_children FROM pairs
  UNION ALL
  SELECT p.work_id, c.ancestor_id, TRUE FROM pairs p JOIN child_of c ON p.institution_id = c.institution_id
  UNION ALL
  SELECT p.work_id, l.ancestor_id, FALSE FROM pairs p JOIN lineage_of l ON p.institution_id = l.institution_id
),
per_work AS (
  SELECT work_id, counted_id, BOOL_OR(via_children) AS via_children
  FROM expanded
  GROUP BY work_id, counted_id
)
SELECT
  counted_id AS institution_id,
  CAST(COUNT_IF(via_children) AS INT) AS works_count_with_children,
  CAST(COUNT(*) AS INT) AS works_count_with_children_and_predecessors
FROM per_work
GROUP BY counted_id


In [ ]:
%sql
-- Guardrail: the table must cover most institutions with works (about 110K in Oct 2026).
SELECT CASE
  WHEN (SELECT COUNT(*) FROM openalex.institutions.institution_lineage_works_counts) < 50000
  THEN RAISE_ERROR('GUARDRAIL FAILED: institution_lineage_works_counts has fewer than 50,000 rows.')
END


In [ ]:
%sql
-- Verification (#1823): INRAE (4210088668) on 26 Oct 2026: direct 217,301; with children 584,623 (407 children);
-- with children and predecessors (INRA 4387154544, Irstea 4387154183) 620,754 = the API's lineage filter count.
SELECT * FROM openalex.institutions.institution_lineage_works_counts WHERE institution_id = 4210088668
